# Day 4

## Tokenizing with code

In [4]:
import tiktoken

encoding = tiktoken.encoding_for_model("gpt-4.1-mini")

tokens = encoding.encode("Hi my name is Ed and I like banoffee pie")

In [5]:
tokens

[12194, 922, 1308, 382, 6117, 326, 357, 1299, 9171, 26458, 5148]

In [6]:
for token_id in tokens:
    token_text = encoding.decode([token_id])
    print(f"{token_id} = {token_text}")

12194 = Hi
922 =  my
1308 =  name
382 =  is
6117 =  Ed
326 =  and
357 =  I
1299 =  like
9171 =  ban
26458 = offee
5148 =  pie


In [7]:
encoding.decode([326])

' and'

# And another topic!

### The Illusion of "memory"

Many of you will know this already. But for those that don't -- this might be an "AHA" moment!

In [28]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)
api_key = os.getenv('GROQ_API_KEY')

if not api_key:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
elif not api_key.startswith("sk-proj-"):
    print("An API key was found, but it doesn't start sk-proj-; please check you're using the right key - see troubleshooting notebook")
else:
    print("API key found and looks good so far!")

An API key was found, but it doesn't start sk-proj-; please check you're using the right key - see troubleshooting notebook


### You should be very comfortable with what the next cell is doing!

_I'm creating a new instance of the OpenAI Python Client library, a lightweight wrapper around making HTTP calls to an endpoint for calling the GPT LLM, or other LLM providers_

In [29]:
from openai import OpenAI
GROQ_BASE_URL="https://api.groq.com/openai/v1"
openai = OpenAI(base_url=GROQ_BASE_URL,api_key=api_key)
MODEL="llama-3.3-70b-versatile"

### A message to OpenAI is a list of dicts

In [32]:
messages = [
    {"role": "system", "content": "You are a helpful assistant"},
    {"role": "user", "content": "Hi! I'm Drona!"}
    ]

In [33]:
response = openai.chat.completions.create(model=MODEL, messages=messages)
response.choices[0].message.content

"Hello Drona, it's nice to meet you. Is there something I can help you with or would you like to chat?"

### OK let's now ask a follow-up question

In [34]:
messages = [
    {"role": "system", "content": "You are a helpful assistant"},
    {"role": "user", "content": "What's my name?"}
    ]

In [35]:
response = openai.chat.completions.create(model=MODEL, messages=messages)
response.choices[0].message.content

"I don't know your name. I'm a large language model, I don't have the ability to recall personal information about individuals, including their names. Each time you interact with me, it's a new conversation and I don't retain any information from previous conversations. If you'd like to share your name with me, I'd be happy to chat with you and address you by your name."

### Wait, wha??

We just told you!

What's going on??

Here's the thing: every call to an LLM is completely STATELESS. It's a totally new call, every single time. As AI engineers, it's OUR JOB to devise techniques to give the impression that the LLM has a "memory".

In [36]:
messages = [
    {"role": "system", "content": "You are a helpful assistant"},
    {"role": "user", "content": "Hi! I'm Drona!"},
    {"role": "assistant", "content": "Hi Drona! How can I assist you today?"},
    {"role": "user", "content": "What's my name?"}
    ]

In [37]:
response = openai.chat.completions.create(model=MODEL, messages=messages)
response.choices[0].message.content

'Your name is Drona.'

## To recap

With apologies if this is obvious to you - but it's still good to reinforce:

1. Every call to an LLM is stateless
2. We pass in the entire conversation so far in the input prompt, every time
3. This gives the illusion that the LLM has memory - it apparently keeps the context of the conversation
4. But this is a trick; it's a by-product of providing the entire conversation, every time
5. An LLM just predicts the most likely next tokens in the sequence; if that sequence contains "My name is Ed" and later "What's my name?" then it will predict.. Ed!

The ChatGPT product uses exactly this trick - every time you send a message, it's the entire conversation that gets passed in.

"Does that mean we have to pay extra each time for all the conversation so far"

For sure it does. And that's what we WANT. We want the LLM to predict the next tokens in the sequence, looking back on the entire conversation. We want that compute to happen, so we need to pay the electricity bill for it!

